# Descriptive Analysis

Phase 1 capstone. Five zones (BE, FI, SG, US-MIDA-PJM, US-NY-NYIS), hourly data 2021-01 through 2025-12. Read-only over `data/raw/`; produces figures bound for the report's Data section and the per-region heterogeneity atlas (Contribution 3).

## 1. Setup

In [ ]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import carbon_forecast
from carbon_forecast.data.storage import read_parquet
from carbon_forecast.data.zones import ZONES
from carbon_forecast.plotting.config import (
    BG_COLOR,
    CI_COLOR,
    discrete_percentile_colorscale,
    ENERGY_PALETTE,
    ENERGY_SOURCE_ORDER,
    PLOT_H,
    PLOT_W,
    REGIONAL_PALETTE,
    FLOW_IMPORT_COLOR,
    FLOW_EXPORT_COLOR,
    apply_defaults,
    style_fig,
)

apply_defaults()

PROJECT_ROOT = Path(carbon_forecast.__file__).resolve().parents[2]
DATA_ROOT = PROJECT_ROOT / "data"
ZONE_KEYS = [z.em_key for z in ZONES]

print("Zones    :", ZONE_KEYS)
print("Data root:", DATA_ROOT)


## 2. Long-format master frames

One loader per (zone, endpoint) family. Each concatenates every monthly Parquet on disk into one long DataFrame indexed by UTC time. Cached in memory via `lru_cache` so subsequent cells reuse the same frames.

In [ ]:
def _load_endpoint(zone: str, endpoint: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/em" / zone / endpoint
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_ci_long() -> pd.DataFrame:
    frames = []
    for zone in ZONE_KEYS:
        df = _load_endpoint(zone, "carbon-intensity/past-range")
        if df.empty:
            continue
        df = df.copy()
        df["zone"] = zone
        frames.append(df)
    return pd.concat(frames) if frames else pd.DataFrame()


@lru_cache(maxsize=None)
def load_pb(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "power-breakdown/past-range")


@lru_cache(maxsize=None)
def load_flows(zone: str) -> pd.DataFrame:
    return _load_endpoint(zone, "electricity-flows/past-range")


@lru_cache(maxsize=None)
def load_weather(zone: str) -> pd.DataFrame:
    root = DATA_ROOT / "raw/weather" / zone
    if not root.exists():
        return pd.DataFrame()
    frames = [read_parquet(f) for f in sorted(root.glob("*.parquet"))]
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

In [ ]:
ci_long = load_ci_long()
print("CI long shape :", ci_long.shape)
print("Zones present :", sorted(ci_long["zone"].unique()))
print("Date range    :", ci_long.index.min(), "to", ci_long.index.max())
ci_long.head()

## 3. Analysis helpers

Daily-mean resampling (the granularity for the time-series section), an energy-source to high-level group map (fossil / nuclear / renewable / unknown), and a muted-orange colorscale for the heatmaps.

In [ ]:
# Energy source -> high-level group for the production overview.
SOURCE_GROUP = {
    "nuclear": "nuclear",
    "battery_discharge": "renewable",
    "hydro_discharge": "renewable",
    "hydro": "renewable",
    "wind": "renewable",
    "biomass": "renewable",
    "solar": "renewable",
    "geothermal": "renewable",
    "gas": "fossil",
    "coal": "fossil",
    "oil": "fossil",
    "unknown": "unknown",
}
GROUP_ORDER = ["nuclear", "renewable", "fossil", "unknown"]
GROUP_PALETTE = {
    "nuclear":   "#493C6B",
    "renewable": "#2E9956",
    "fossil":    "#C0504D",
    "unknown":   "#A69C9C",
}

# Muted orange sequential scale for the CI heatmaps.
MUTED_ORANGE = [[0.0, "#FBF1E6"], [0.5, "#E8B07A"], [1.0, "#B5651D"]]


def daily_ci(zone: str) -> pd.Series:
    """Daily-mean carbon intensity for one zone."""
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    return s.resample("1D").mean()


def daily_prod_by_source(zone: str) -> pd.DataFrame:
    """Daily-mean production (MW) per source, columns in canonical order."""
    pb = load_pb(zone)
    if pb.empty:
        return pd.DataFrame()
    prod_cols = [c for c in pb.columns if c.startswith("prod_") and c.endswith("_mw")]
    src = pb[prod_cols].copy()
    src.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]
    daily = src.resample("1D").mean()
    return daily.reindex(columns=[c for c in ENERGY_SOURCE_ORDER if c in daily.columns])


def daily_prod_by_group(zone: str) -> pd.DataFrame:
    """Daily-mean production (MW) aggregated into the four high-level groups."""
    daily = daily_prod_by_source(zone)
    if daily.empty:
        return daily
    grouped = {}
    for grp in GROUP_ORDER:
        cols = [c for c in daily.columns if SOURCE_GROUP.get(c) == grp]
        if cols:
            grouped[grp] = daily[cols].sum(axis=1)
    return pd.DataFrame(grouped)


def daily_net_flow(zone: str) -> pd.Series:
    """Daily-mean net imports (import_total - export_total), MW."""
    pb = load_pb(zone)
    if pb.empty or "import_total_mw" not in pb.columns or "export_total_mw" not in pb.columns:
        return pd.Series(dtype=float)
    net = pb["import_total_mw"].fillna(0.0) - pb["export_total_mw"].fillna(0.0)
    return net.resample("1D").mean()

# Section 1 — Initial time-series exploration

All charts use **daily-mean** values over the full 2021 to 2025 window (about 1,826 points per region), which keeps multi-year trend and seasonality legible while letting bar overlays read cleanly. Regions are stacked vertically (5 rows, shared time axis) for at-a-glance comparison.

### 1. Baseline carbon intensity

Daily-mean carbon intensity per region. The reference series everything else overlays onto.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.05,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = daily_ci(zone)
    fig.add_trace(
        go.Scatter(x=s.index, y=s.values, mode="lines",
                   line=dict(color=REGIONAL_PALETTE[zone], width=1),
                   name=zone, showlegend=False),
        row=i, col=1,
    )
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1)
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Carbon intensity daily mean by region", subtitle="January 2021 to December 2025", height=1250)
fig.show()

### 2. Carbon intensity over production mix

Daily-mean production (MW, left axis) stacked into fossil / nuclear / renewable / unknown, with carbon intensity (gCO2eq/kWh, right axis) overlaid as a dark line. Bars are semi-transparent so the CI line reads through. Where renewables and nuclear dominate the stack, CI should sit low; where fossil grows, CI should rise.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.05,
    specs=[[{"secondary_y": True}] for _ in ZONE_KEYS],
)
seen = set()
for i, zone in enumerate(ZONE_KEYS, start=1):
    grouped = daily_prod_by_group(zone)
    for grp in GROUP_ORDER:
        if grp not in grouped.columns:
            continue
        show = grp not in seen
        seen.add(grp)
        fig.add_trace(
            go.Bar(
                x=grouped.index, 
                y=grouped[grp], 
                name=grp,
                marker_color=GROUP_PALETTE[grp], 
                marker_line_width=0,
                opacity=0.50, 
                legendgroup=grp, 
                showlegend=show),
            row=i, 
            col=1, 
            secondary_y=False,
     
        )
    ci = daily_ci(zone)
    fig.add_trace(
        go.Scatter(
            x=ci.index, 
            y=ci.values, 
            mode="lines",
            line=dict(color = "#000000", width = 0.75),
        opacity = 0.5,
        name="carbon intensity", 
        legendgroup="ci", 
        showlegend=(i == 1)),
        row=i, col=1, secondary_y=True,
    )
    fig.update_yaxes(title_text="MW", row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1, secondary_y=True)
fig.update_layout(barmode="stack", barnorm="percent")
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(fig, "Generation mix and carbon intensity by region", subtitle="Share of total energy generation by renewable, fossil and nuclear against total carbon intensity per region", height=1500)
fig.update_yaxes(showgrid=False, secondary_y=True)  # avoid double gridlines
fig.show()

### 3. Production by source

Daily-mean production per source, one line per source, per region. Shows how each fuel's contribution moves over the five years (wind growth, nuclear outages, seasonal hydro and solar).

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.06,
)
seen = set()
for i, zone in enumerate(ZONE_KEYS, start=1):
    daily = daily_prod_by_source(zone)
    for source in daily.columns:
        show = source not in seen
        seen.add(source)
        fig.add_trace(
            go.Scatter(x=daily.index, y=daily[source], mode="lines",
                       line=dict(color=ENERGY_PALETTE.get(source, "#888"), width=1),
                       name=source, legendgroup=source, showlegend=show),
            row=i, col=1,
        )
    fig.update_yaxes(title_text="MW", row=i, col=1)
fig.update_xaxes(title_text="Date", row=len(ZONE_KEYS), col=1)
style_fig(
    fig, 
    "Energy Generation by source over time, daily mean",
    height=1200)
fig.show()

### 4. Carbon intensity over net cross-border flow

Daily-mean net imports (MW, left axis; positive = importing, negative = exporting) as semi-transparent bars, with carbon intensity (right axis) overlaid. Reveals whether a region leans on imports and whether import-heavy periods coincide with lower or higher CI.

In [ ]:
fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1, shared_xaxes=True,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.06,
    specs=[[{"secondary_y": True}] for _ in ZONE_KEYS],
)

for i, zone in enumerate(ZONE_KEYS, start=1):

    # Net imports-exports bar chart, color-coded
    net = daily_net_flow(zone)
    colors = [FLOW_IMPORT_COLOR if val >= 0 else FLOW_EXPORT_COLOR for val in net.values]

    fig.add_trace(

        go.Bar(
            x = net.index, 
            y = net.values, 
            name = "net imports",
            marker_color = colors, 
            marker_line_width = 0, 
            opacity = 0.5,
            legendgroup = "net", 
            showlegend = (i == 1)),
            row = i, 
            col = 1, 
            secondary_y = False,
    )

    # Line chart for carbon intensity
    ci = daily_ci(zone)

    fig.add_trace(
        go.Scatter(
            x=ci.index, 
            y=ci.values, 
            mode="lines",
            line=dict(color="#000000", width=0.5),
            name="carbon intensity", 
            opacity = 0.5,
            legendgroup="ci", 
            showlegend=(i == 1)),
            row=i, 
            col=1, 
            secondary_y=True,
    )

    fig.update_yaxes(title_text="MW", row = i, col = 1, secondary_y = False)
    fig.update_yaxes(title_text="gCO2eq/kWh", row=i, col=1, secondary_y=True)

fig.update_xaxes(
    title_text="Date", 
    row=len(ZONE_KEYS), 
    col=1)

style_fig(
    fig, 
    title="Carbon Intensity & Net Enegy Flow displays stronger corr. in North America",
    subtitle = "Net cross-border energy flow and carbon intensity by region, 2021 - 2025.",
    height=1200)

fig.update_yaxes(
    showgrid = False, 
    secondary_y=True)
    
fig.show()

# Section 2 — Distributions

How carbon intensity and the production mix are distributed: across the value range, across the calendar year, and across the daily and weekly cycle.

### 5. Carbon intensity distribution

Overlapping histograms, one per region, semi-transparent. Reveals the shape and spread of each region's CI: FI low and right-skewed, SG high and narrow, BE wide, the US zones in between.

In [ ]:
fig = px.histogram(
    ci_long,
    x="carbon_intensity_gco2eq_kwh",
    color="zone",
    color_discrete_map=REGIONAL_PALETTE,
    category_orders={"zone": ZONE_KEYS},
    barmode="overlay",
    opacity=0.55,
    nbins=80,
    labels={"carbon_intensity_gco2eq_kwh": "Carbon Intensity (gCO2eq/kWh)", "zone": "Zone"},
)
fig.update_yaxes(title_text="Count")
style_fig(
    fig, 
    "Singapore carbon intensity concentrates between 490 - 499 gCO2eq/kWh",
    subtitle = "Carbon intensity distribution by region (2021 - 2025)"),
fig.show()

### 6. Annual production mix, share of total

Each region's annual production normalised to 100 percent, so composition is comparable despite zone scales differing by an order of magnitude. Baseload (nuclear) at the bottom, fossils on top.

In [ ]:
records = []
for zone in ZONE_KEYS:
    pb = load_pb(zone)
    if pb.empty:
        continue
    prod_cols = [c for c in pb.columns if c.startswith("prod_") and c.endswith("_mw")]
    src = pb[prod_cols].copy()
    src.columns = [c[len("prod_"):-len("_mw")] for c in prod_cols]
    annual = src.groupby(src.index.year).mean()
    for year in annual.index:
        for source in annual.columns:
            val = annual.loc[year, source]
            if pd.notna(val):
                records.append({"zone": zone, "year": int(year), "source": source, "mw": val})
long_prod = pd.DataFrame(records)
long_prod["pct"] = (
    100.0 * long_prod["mw"]
    / long_prod.groupby(["zone", "year"])["mw"].transform("sum")
)

fig = px.bar(
    long_prod,
    x="year", y="pct", color="source",
    facet_col="zone", facet_col_spacing=0.044,
    color_discrete_map=ENERGY_PALETTE,
    category_orders={"source": list(ENERGY_SOURCE_ORDER), "zone": ZONE_KEYS},
    labels={"pct": "Share of Production (%)", "year": "Year", "source": "Source"},
)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.update_yaxes(range=[0, 100], ticksuffix="%")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(barmode="stack")
style_fig(
    fig,
    "European regions display a balanced grid, SG and the US are highly dependent on Gas",
    subtitle="Energy Generation Mix by Region (2021 - 2025)",
    width=int(PLOT_W * 1.6),
)
fig.show()

### 7. Carbon intensity by day of week and hour of day

One heatmap per region (stacked vertically), hours of day as columns and days of week as rows, square cells. Each region is scaled to its own range so the within-region weekly and daily pattern is visible regardless of absolute CI level.

In [ ]:
DAY_LABELS = ["M", "T", "W", "T", "F", "S", "S"]  # Mon..Sun (numeric x avoids label collision)
HOUR_LABELS = [f"{h:02d}:00" for h in range(24)]   # military time

cscale = discrete_percentile_colorscale(n_bins=20)
mid = (len(ZONE_KEYS) + 1) // 2
fig = make_subplots(
    rows=1, cols=len(ZONE_KEYS),
    subplot_titles=ZONE_KEYS, horizontal_spacing=0.04,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    # Rows = hour of day, columns = day of week.
    g = (
        s.to_frame("ci")
        .assign(dow=s.index.dayofweek, hod=s.index.hour)
        .groupby(["hod", "dow"])["ci"].mean().unstack("dow")
    )
    # Independent per-region percentile: rank each cell within this region only.
    own = np.sort(g.values.ravel())
    own = own[~np.isnan(own)]
    z_pct = np.searchsorted(own, g.values, side="right") / len(own) * 100
    fig.add_trace(
        go.Heatmap(
            z=z_pct,
            x=list(g.columns),
            y=[HOUR_LABELS[h] for h in g.index],
            customdata=g.values,
            hovertemplate="Hour %{y}<br>CI %{customdata:.0f} gCO2eq/kWh<br>Percentile %{z:.0f}%<extra></extra>",
            colorscale=cscale, zmin=0, zmax=100,
            showscale=(i == len(ZONE_KEYS)),
            colorbar=dict(
                orientation="h", x=0.5, xanchor="center", y=-0.16, yanchor="top",
                thickness=14, len=0.7, tickvals=[0, 25, 50, 75, 100], ticksuffix="%",
                title=dict(text="Carbon Intensity Percentile (Per Region)", side="bottom"),
            ),
        ),
        row=1, col=i,
    )
    fig.update_xaxes(tickmode="array", tickvals=list(g.columns),
                     ticktext=[DAY_LABELS[d] for d in g.columns], tickangle=0, row=1, col=i)
    if i == 1:
        fig.update_yaxes(title_text="Hour", row=1, col=i)
    else:
        fig.update_yaxes(showticklabels=False, row=1, col=i)
    if i == mid:
        fig.update_xaxes(title_text="Day of Week", row=1, col=i)
style_fig(fig, "Carbon intensity by day of week and hour of day, per region", subtitle="Color is each region's own carbon-intensity percentile, one hue per 5 percentile", width=1100, height=720)
fig.update_layout(margin_b=180)
# Square cells; 00:00 at the top of every panel.
for i in range(1, len(ZONE_KEYS) + 1):
    xref = "x" if i == 1 else f"x{i}"
    fig.update_yaxes(scaleanchor=xref, scaleratio=1, autorange="reversed", row=1, col=i)
fig.show()

### 8. Carbon intensity by month and year

One heatmap per region (stacked vertically), months as columns and years as rows, square cells, same muted-orange scale. Reveals year-over-year drift and the seasonal band within each year.

In [ ]:
MONTH_LABELS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

cscale = discrete_percentile_colorscale(n_bins=20)
mid = (len(ZONE_KEYS) + 1) // 2
fig = make_subplots(
    rows=1, cols=len(ZONE_KEYS),
    subplot_titles=ZONE_KEYS, horizontal_spacing=0.04,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    s = ci_long[ci_long["zone"] == zone]["carbon_intensity_gco2eq_kwh"]
    # Rows = month, columns = year.
    g = (
        s.to_frame("ci")
        .assign(year=s.index.year, month=s.index.month)
        .groupby(["month", "year"])["ci"].mean().unstack("year")
    )
    # Independent per-region percentile: rank each cell within this region only.
    own = np.sort(g.values.ravel())
    own = own[~np.isnan(own)]
    z_pct = np.searchsorted(own, g.values, side="right") / len(own) * 100
    fig.add_trace(
        go.Heatmap(
            z=z_pct,
            x=[f"{y % 100}'" for y in g.columns],
            y=[MONTH_LABELS[m - 1] for m in g.index],
            customdata=g.values,
            hovertemplate="%{y}<br>CI %{customdata:.0f} gCO2eq/kWh<br>Percentile %{z:.0f}%<extra></extra>",
            colorscale=cscale, zmin=0, zmax=100,
            showscale=(i == len(ZONE_KEYS)),
            colorbar=dict(
                orientation="h", x=0.5, xanchor="center", y=-0.16, yanchor="top",
                thickness=14, len=0.7, tickvals=[0, 25, 50, 75, 100], ticksuffix="%",
                title=dict(text="Carbon Intensity Percentile (Per Region)", side="bottom"),
            ),
        ),
        row=1, col=i,
    )
    if i == 1:
        fig.update_yaxes(title_text="Month", row=1, col=i)
    else:
        fig.update_yaxes(showticklabels=False, row=1, col=i)
    if i == mid:
        fig.update_xaxes(title_text="Year", row=1, col=i)
style_fig(fig, "Carbon intensity by month and year, per region", subtitle="Color is each region's own carbon-intensity percentile, one hue per 5 percentile", width=1000, height=620)
fig.update_layout(margin_b=180)
# Square cells; Jan at the top of every panel.
for i in range(1, len(ZONE_KEYS) + 1):
    xref = "x" if i == 1 else f"x{i}"
    fig.update_yaxes(scaleanchor=xref, scaleratio=1, autorange="reversed", row=1, col=i)
fig.show()

# Section 3 — Correlation

How carbon intensity co-moves with its candidate drivers. First a scatter matrix of CI against five engineered drivers per region, then a full correlation matrix over every raw numeric variable per region. Daily means over 2021 to 2025 throughout, matching the earlier sections.

### 9. Carbon intensity against its drivers

Scatter matrix: one column per region, one row per candidate driver. The y axis is always carbon intensity (gCO2eq/kWh); the x axis changes by row. Drivers: total generation (MW), net cross-border flow (imports minus exports, MW), renewables share of generation (%), average temperature, and average wind speed. All points use the carbon-intensity color. Reading down a column shows which drivers track CI in that region; reading across a row shows how the same relationship shifts between regions.

In [ ]:
# Per-region daily features: the candidate CI drivers, all as daily means.
def daily_features(zone: str) -> pd.DataFrame:
    grouped = daily_prod_by_group(zone)
    total = grouped.sum(axis=1)
    renew = grouped["renewable"] if "renewable" in grouped.columns else pd.Series(0.0, index=grouped.index)
    w = load_weather(zone)
    temp = w["temperature_2m"].resample("1D").mean() if not w.empty else pd.Series(dtype=float)
    wind = w["wind_speed_10m"].resample("1D").mean() if not w.empty else pd.Series(dtype=float)
    return pd.DataFrame({
        "ci": daily_ci(zone),
        "total_gen": total,
        "net_flow": daily_net_flow(zone),
        "renew_share": renew / total * 100,
        "temp": temp,
        "wind": wind,
    })


METRIC_ROWS = [
    ("total_gen",   "Total Generation (MW)"),
    ("net_flow",    "Net Flow (MW)"),
    ("renew_share", "Renewables Share (%)"),
    ("temp",        "Temperature (\u00b0C)"),
    ("wind",        "Wind Speed (m/s)"),
]

feats = {z: daily_features(z) for z in ZONE_KEYS}

fig = make_subplots(
    rows=len(METRIC_ROWS), cols=len(ZONE_KEYS),
    column_titles=ZONE_KEYS,
    row_titles=[lbl for _, lbl in METRIC_ROWS],
    horizontal_spacing=0.025, vertical_spacing=0.035,
)
for r, (col, _lbl) in enumerate(METRIC_ROWS, start=1):
    for c, zone in enumerate(ZONE_KEYS, start=1):
        df = feats[zone]
        fig.add_trace(
            go.Scattergl(
                x=df[col], y=df["ci"], mode="markers",
                marker=dict(color=CI_COLOR, size=3, opacity=0.35),
                showlegend=False,
            ),
            row=r, col=c,
        )
    fig.update_yaxes(title_text="gCO2eq/kWh", row=r, col=1)
style_fig(
    fig,
    "Carbon intensity against its candidate drivers, per region",
    subtitle="Daily means 2021 to 2025. Columns are regions, rows are drivers (x) versus carbon intensity (y)",
    width=1300, height=1500,
)
fig.show()

### 10. Correlation matrix of all numeric variables

One square Pearson correlation matrix per region (stacked vertically so the labels stay readable), over every raw numeric variable: each production source (P), each consumption source (C), total imports and exports, the six weather variables, and carbon intensity itself. Carbon intensity is pinned to the first row and column. Cells are labelled to two decimals. Color is diverging and anchored at zero: carbon-intensity orange for strong positive correlation, cool blue for strong negative, neutral near zero. Constant or empty columns are dropped per region, so the variable set varies slightly by zone.

In [ ]:
DIVERGING_CORR = [[0.0, FLOW_EXPORT_COLOR], [0.5, BG_COLOR], [1.0, CI_COLOR]]
CI_COL = "carbon_intensity_gco2eq_kwh"


def daily_numeric_all(zone: str) -> pd.DataFrame:
    """All raw numeric columns (power-breakdown + weather + CI) as daily means.
    Drops all-NaN and constant columns so they do not produce empty corr cells."""
    parts = []
    pb = load_pb(zone)
    if not pb.empty:
        parts.append(pb.resample("1D").mean())
    w = load_weather(zone)
    if not w.empty:
        parts.append(w.resample("1D").mean())
    parts.append(daily_ci(zone).rename(CI_COL))
    df = pd.concat(parts, axis=1)
    keep = df.columns[(df.notna().sum() > 1) & (df.nunique() > 1)]
    return df[keep]


def short_label(c: str) -> str:
    if c == CI_COL:
        return "CI"
    if c.startswith("prod_") and c.endswith("_mw"):
        return c[5:-3] + " (P)"
    if c.startswith("cons_") and c.endswith("_mw"):
        return c[5:-3] + " (C)"
    if c == "import_total_mw":
        return "import"
    if c == "export_total_mw":
        return "export"
    return c


fig = make_subplots(
    rows=len(ZONE_KEYS), cols=1,
    subplot_titles=ZONE_KEYS, vertical_spacing=0.02,
)
for i, zone in enumerate(ZONE_KEYS, start=1):
    df = daily_numeric_all(zone)
    cols = [CI_COL] + [c for c in df.columns if c != CI_COL]
    corr = df[cols].corr()
    labels = [short_label(c) for c in cols]
    fig.add_trace(
        go.Heatmap(
            z=corr.values, x=labels, y=labels,
            texttemplate="%{z:.2f}", textfont=dict(size=7),
            colorscale=DIVERGING_CORR, zmin=-1, zmid=0, zmax=1,
            showscale=(i == 1),
            colorbar=dict(
                orientation="v", x=1.02, xanchor="left", y=1.0, yanchor="top",
                thickness=14, len=0.18, tickvals=[-1, -0.5, 0, 0.5, 1],
                title=dict(text="Pearson r", side="right"),
            ),
            hovertemplate="%{y} vs %{x}<br>r %{z:.2f}<extra></extra>",
        ),
        row=i, col=1,
    )
style_fig(
    fig,
    "Correlation of all numeric variables with carbon intensity, per region",
    subtitle="Pearson r on daily means 2021 to 2025. (P) production, (C) consumption. CI is the first row and column",
    width=900, height=4200,
)
# Square cells; CI pinned top-left, variable order matches across both axes.
for i in range(1, len(ZONE_KEYS) + 1):
    xref = "x" if i == 1 else f"x{i}"
    fig.update_yaxes(scaleanchor=xref, scaleratio=1, autorange="reversed", row=i, col=1)
    fig.update_xaxes(tickfont=dict(size=8), tickangle=90, row=i, col=1)
    fig.update_yaxes(tickfont=dict(size=8), row=i, col=1)
fig.show()